# Лабораторная работа 7. Обработка данных из файла. Использование коллекций

ФИО студента: Понкратов Владимир Александрович

Группа студента: ИНБ-б-о-26-2

Вариант: 4

## Задание

Дан файл `p7_data_04.csv`, содержащий информацию о сотрудниках компаний. Поля файла: `ID` – уникальный ключ записи; `FName`, `LName` – имя и фамилия сотрудника; `Company` – компания; `Salary` – заработная плата (записана со знаком `$`); `Language` – иностранный язык, которым владеет сотрудник.

Необходимо ответить на следующие вопросы:

1. Количество сотрудников в каждой из 4-х компаний;
2. Среднюю з/п в каждой из 4-х компаний;
3. Количество сотрудников, владеющих английским языком;
4. Количество сотрудников, получающих более $3000,00.

## Анализ задания, получение данных

Файл `p7_data_04.csv` нужно поместить в рабочую директорию (в Google Colab он загружается через панель «Файлы» слева либо скачивается командой `!wget`, если известна ссылка на файл в репозитории). Если файл уже загружен — следующую ячейку можно просто выполнить для проверки, ничего скачивать не нужно.

In [ ]:
import os

print('Файл на месте:', os.path.exists('p7_data_04.csv'))

Реализуем чтение всего файла построчно. Используем метод `open()`, а затем будем обходить файл как коллекцию с использованием цикла `for line in F`. Csv-файл используется в данной работе как обычный текстовый файл, и в цикле `for` производится последовательный обход всех его строк.

В файле 1000 строк с данными, поэтому будем печатать только первые `n_to_print` строк, но обрабатывать будем все (чтобы не возникло скрытых ошибок).

In [ ]:
F = open('p7_data_04.csv')
n_to_print = 10
for line in F:
    n_to_print -= 1
    if n_to_print > 0:
        print(line)

F.close()

Строки перебираются, но теперь необходимо разделить единую строку на отдельные элементы данных. Для этого выполним метод `split(sep)`, который разрезает строку, используя символ `sep` в качестве разделителя. Также обратим внимание на тот факт, что в первой строке содержатся не данные, а заголовки столбцов.

In [ ]:
# доработка предыдущего кода
F = open('p7_data_04.csv')
header = F.readline().strip().split(',')
print(header)
print('-' * 20)
n_to_print = 10
for line in F:
    # производим обработку в любом случае ...
    data = line.strip().split(',')
    # ... а печатаем только первые n_to_print
    n_to_print -= 1
    if n_to_print > 0:
        print(data)

F.close()

Метод `split()` вернул список отдельных строк — полей исходной строки. Теперь необходимо сформировать структуру данных, максимально полно описывающую предметную область.

Для этого сформируем список словарей: все данные в процессе чтения будем построчно перемещать в список `all_clients`, при этом каждый сотрудник будет представлен в виде словаря. Например, первая строка файла будет выглядеть следующим образом:

`example = {'ID': '67-9921732', 'FName': 'Giorgi', 'LName': 'Kirvell', 'Company': 'Zuko Ltd', 'Salary': 1018.20, 'Language': 'Italian'}`

то есть будет сформирована структура данных, в которой все элементы представляют собой пары вида `ключ : значение`. Обратиться к элементу можно по ключу, например `example['Salary']`.

Обратите внимание: зарплата записана в виде строки со знаком доллара (`'$1018.20'`), поэтому перед вычислениями её нужно очистить от символа `$` и привести к числу методом `float()`. Имена сотрудников (`FName`, `LName`) оставляем строками.

In [ ]:
# Чтение файла и формирование списка словарей
F = open('p7_data_04.csv')
header = F.readline().strip().split(',')

# список всех сотрудников
all_clients = []

for line in F:
    data = line.strip().split(',')
    client = {}  # создаем пустой словарь
    for i, key in enumerate(header):  # обходим все заголовки
        client[key] = data[i]  # добавляем в словарь пару ключ:значение

    # некоторые значения конвертируем к подходящим типам
    # '$1018.20' -> 1018.2
    client['Salary'] = float(client['Salary'].replace('$', ''))

    all_clients.append(client)  # добавляем сотрудника в список

F.close()  # файл больше не нужен

print('Всего сотрудников:', len(all_clients))
print(all_clients[:3])  # напечатаем первых 3 сотрудников

## Решение задачи 1

Найдем количество сотрудников в каждой из 4-х компаний.

Переберем все элементы списка `all_clients` и для каждой компании будем накапливать счетчик в словаре `count_by_company`. При первом появлении компании создадим для неё счетчик, равный 1, при повторном — увеличим его на 1.

In [ ]:
# Задача 1: количество сотрудников в каждой компании
count_by_company = {}
for client in all_clients:
    company = client['Company']
    if company in count_by_company:
        count_by_company[company] += 1
    else:
        count_by_company[company] = 1

for company in sorted(count_by_company):
    print('   %s: %d сотрудников' % (company, count_by_company[company]))

## Решение задачи 2

Найдем среднюю заработную плату в каждой из 4-х компаний.

Для этого параллельно с подсчетом количества сотрудников накопим в словаре `sum_by_company` сумму зарплат сотрудников каждой компании. Средняя зарплата = сумма зарплат компании / количество сотрудников компании.

In [ ]:
# Задача 2: средняя з/п в каждой компании
sum_by_company = {}
for client in all_clients:
    company = client['Company']
    if company in sum_by_company:
        sum_by_company[company] += client['Salary']
    else:
        sum_by_company[company] = client['Salary']

for company in sorted(sum_by_company):
    avg = sum_by_company[company] / count_by_company[company]
    print('   %s: $%.2f' % (company, avg))

## Решение задачи 3

Найдем количество сотрудников, владеющих английским языком.

Переберем всех сотрудников и увеличим счетчик `counter_eng`, если поле `Language` равно строке `'English'`.

In [ ]:
# Задача 3: количество сотрудников, владеющих английским языком
counter_eng = 0
for client in all_clients:
    if client['Language'] == 'English':
        counter_eng += 1

print('Сотрудников, владеющих английским языком:', counter_eng)

## Решение задачи 4

Найдем количество сотрудников, получающих более $3000,00.

Так как зарплаты уже приведены к числовому типу, достаточно перебрать список и сравнить поле `Salary` с числом 3000. Сравнение строгое (`>`), то есть сотрудник с зарплатой ровно $3000,00 в подсчет не попадает.

In [ ]:
# Задача 4: количество сотрудников с зарплатой более $3000,00
counter_3000 = 0
for client in all_clients:
    if client['Salary'] > 3000:
        counter_3000 += 1

print('Сотрудников с зарплатой более $3000,00:', counter_3000)

## Ответы

1. Количество сотрудников: **Craft Ltd — 242, NodaTLT Co — 246, Workus Co — 257, Zuko Ltd — 255** (всего 1000).
2. Средняя з/п: **Craft Ltd — $3135,37; NodaTLT Co — $2967,03; Workus Co — $2965,15; Zuko Ltd — $2991,64**.
3. Сотрудников, владеющих английским языком: **250**.
4. Сотрудников, получающих более $3000,00: **497**.

## Выводы

В данной работе был реализован построчный разбор csv-файла: строки читались методом `open()` и циклом `for line in F`, разделялись методом `split(',')` и преобразовывались в список словарей `all_clients`, что позволило обращаться к каждому полю записи по имени. Для числового поля `Salary` выполнена конвертация типа: удаление символа `$` и приведение к `float`. С помощью оператора `for ... in` и условного оператора `if` решены четыре задачи: подсчет сотрудников по компаниям (словарь-счетчик), вычисление средней зарплаты по компаниям (словарь-накопитель сумм), подсчет владеющих английским языком и подсчет сотрудников с зарплатой выше $3000. Применялись функции `open()`, `print()`, `len()`, `float()`, методы строк `strip()`, `split()`, `replace()`, а также тип данных `dict()`, умение работать с которым закреплено в контрольных вопросах.

# Контрольные вопросы

Ответы на контрольные вопросы с кодом.

## 1. Код для печати суммы элементов списка p

In [ ]:
p = [10, 100, 34, 56, 45, 45]
print(sum(p))

**Ответ:** сумма элементов списка вычисляется встроенной функцией `sum()`. Результат: **290**.

## 2. Код для печати количества элементов списка p

In [ ]:
p = [10, 100, 34, 56, 45, 45]
print(len(p))

**Ответ:** количество элементов списка возвращает функция `len()`. Результат: **6**.

## 3. Код для печати суммы элементов, кратных 7

In [ ]:
p = [10, 100, 35, 34, 56, 45, 45, 21]
s = 0
for x in p:
    if x % 7 == 0:  # проверяем кратность 7 (остаток от деления равен 0)
        s += x
print(s)

**Ответ:** кратность 7 проверяется остатком от деления: `x % 7 == 0`. Кратны семи числа 35, 56 и 21, их сумма: **112**.

## 4. Поиск ключей словаря d, для которых значения превышают 45

In [ ]:
d = {4: 10, 12: 100, 35: 3, 34: 56, 56: 11, 45: 'abc'}

result = []
for k in d:
    # значение может быть не числом ('abc'), поэтому проверяем тип
    if isinstance(d[k], (int, float)) and d[k] > 45:
        result.append(k)
print(result)

**Ответ:** перебираем ключи словаря и сравниваем значения с 45. Проверка `isinstance` нужна, чтобы не сравнивать строку `'abc'` с числом (это привело бы к ошибке `TypeError`). Подходят ключи **12** (значение 100) и **34** (значение 56), результат: `[12, 34]`.

## 5. Список квадратов чисел от 1 до 100

In [ ]:
# генерируем список квадратов чисел от 1 до 100 с помощью генератора списка
squares = [x ** 2 for x in range(1, 101)]
print(squares)

**Ответ:** список создается генератором списка `[x ** 2 for x in range(1, 101)]`: `range(1, 101)` дает числа от 1 до 100 включительно, и каждое возводится в квадрат. Результат — 100 элементов: 1, 4, 9, ..., 10000.

## 6. Структура файла csv. Последовательность действий при чтении

**Структура файла csv.** CSV (Comma-Separated Values — «значения, разделенные запятыми») — текстовый формат для представления табличных данных. Файл состоит из строк, каждая строка — одна запись (например, один сотрудник). Первая строка обычно содержит заголовки столбцов — названия полей. Поля внутри строки разделяются разделителем (чаще всего запятой, реже — точкой с запятой или табуляцией). Например, строка нашего файла:

`67-9921732,Giorgi,Kirvell,Zuko Ltd,$1018.20,Italian`

означает: ID = 67-9921732, имя = Giorgi, фамилия = Kirvell, компания = Zuko Ltd, зарплата = $1018.20, язык = Italian. Так как это обычный текстовый файл, данные в нем хранятся как строки, и числа при необходимости нужно преобразовывать в числовой тип.

**Последовательность действий при чтении файла csv:**

1. Открыть файл функцией `open()` и получить файловый объект.
2. Прочитать первую строку методом `readline()` — это заголовки столбцов; удалить символы перевода строки методом `strip()` и разбить по разделителю методом `split(',')`, получив список заголовков.
3. Организовать цикл `for line in F` — последовательный обход всех остальных строк файла.
4. Для каждой строки удалить служебные символы (`strip()`) и разбить ее на поля методом `split(',')`, получив список значений.
5. Сопоставить значения и заголовки (по индексам или через `zip()`) и сформировать словарь, соответствующий одной записи.
6. Привести значения нужных полей к подходящим типам (например, зарплату — к числу).
7. Добавить словарь в список записей.
8. После окончания цикла закрыть файл методом `close()`.

## 7. 'старт': 1472 — что является ключом и что значением

**Ответ:** в паре `'старт': 1472` ключом является строка **'старт'** (она стоит слева от двоеточия), а значением — число **1472** (справа от двоеточия). Ключ служит для доступа к значению, например `d['старт']` вернет 1472.

## 8. Что делает инструкция employee['id'] = 54321

**Ответ:** инструкция присваивает значение 54321 ключу `'id'` словаря `employee`. Если ключа `'id'` в словаре еще не было, он будет создан (в словарь добавится новая пара `'id': 54321`); если ключ уже существовал — его старое значение будет заменено на 54321.

## 9. Что покажет фрагмент кода со словарем stuff

In [ ]:
stuff = {1: 'ааа', 2: 'ббб', 3: 'ввв'}
print(stuff[3])

**Ответ:** код обратится к словарю по ключу 3 и напечатает соответствующее значение: **ввв**.

## 10. Как определить, существует ли пара «ключ : значение» в словаре

**Ответ:** существует ли **ключ**, проверяют оператором `in`: выражение `key in d` возвращает `True`, если ключ есть в словаре, и `False`, если нет:

```python
if 'старт' in d:
    print('Ключ существует')
```

Существует ли конкретная **пара целиком**, проверяют через метод `items()`: он возвращает набор кортежей (ключ, значение), и кортеж можно проверить оператором `in`:

```python
if ('старт', 1472) in d.items():
    print('Пара существует')
```

Прямая проверка значения — `'значение' in d.values()`.

In [ ]:
d = {'старт': 1472, 'финиш': 2000}

# проверка существования ключа
print('старт' in d)              # True

# проверка существования пары целиком
print(('старт', 1472) in d.items())  # True

## 11. Что делает инструкция del inventory[654]

**Ответ:** инструкция удаляет из словаря `inventory` элемент с ключом 654 — то есть пару `654 : значение` полностью исчезает из словаря. Если ключа 654 в словаре нет, возникнет ошибка `KeyError`, поэтому перед удалением ключ обычно проверяют оператором `in`.

## 12. Что покажет фрагмент кода print(len(stuff))

In [ ]:
stuff = {1: 'ааа', 2: 'ббб', 3: 'ввв'}
print(len(stuff))

**Ответ:** функция `len()` возвращает количество пар «ключ : значение» в словаре. Результат: **3**.

## 13. Что покажет фрагмент кода с циклом for k in stuff

In [ ]:
stuff = {1: 'ааа', 2: 'ббб', 3: 'ввв'}
for k in stuff:
    print(k)

**Ответ:** при обходе словаря в цикле `for` переменная `k` поочередно принимает **ключи** словаря (а не значения). Будут напечатаны ключи в порядке их добавления:

```
1
2
3
```

Чтобы получить значения, нужно писать `print(stuff[k])`, а чтобы получить и ключ, и значение — использовать метод `items()`: `for k, v in stuff.items():`.

## 14. Разница между методами pop() и popitem()

**Ответ:**

- `pop(key)` удаляет из словаря элемент **по заданному ключу** и возвращает его значение. Если ключа нет и не указано значение по умолчанию, возникает ошибка `KeyError`. Пример: `v = d.pop('старт')` — удаляет пару и возвращает 1472.
- `popitem()` удаляет из словаря **последний добавленный элемент** (пару целиком) и возвращает его в виде кортежа `(ключ, значение)`. Ключ не указывается. Если словарь пуст, возникает ошибка `KeyError`.

In [ ]:
d = {'старт': 1472, 'финиш': 2000}

# pop() удаляет элемент по ключу и возвращает его значение
v = d.pop('старт')
print('pop вернул:', v, '| словарь после pop:', d)

d = {'старт': 1472, 'финиш': 2000}

# popitem() удаляет последнюю добавленную пару и возвращает её кортежем
item = d.popitem()
print('popitem вернул:', item, '| словарь после popitem:', d)

## 15. Что возвращает метод items()

**Ответ:** метод `items()` возвращает специальный объект-представление (view), содержащий все пары словаря в виде **кортежей (ключ, значение)**. Его удобно обходить в цикле и распаковывать в две переменные:

In [ ]:
stuff = {1: 'ааа', 2: 'ббб', 3: 'ввв'}

print(stuff.items())  # dict_items([(1, 'ааа'), (2, 'ббб'), (3, 'ввв')])

for k, v in stuff.items():
    print(k, '->', v)

## 16. Что возвращают методы keys() и values()

**Ответ:** метод `keys()` возвращает представление (view) всех **ключей** словаря, а метод `values()` — представление всех **значений**. Эти объекты можно обходить в цикле, преобразовывать в список функцией `list()` и проверять принадлежность оператором `in`; изменения в словаре автоматически отражаются в представлении.

In [ ]:
stuff = {1: 'ааа', 2: 'ббб', 3: 'ввв'}

print(stuff.keys())    # dict_keys([1, 2, 3])
print(stuff.values())  # dict_values(['ааа', 'ббб', 'ввв'])

print(list(stuff.keys()))    # [1, 2, 3]
print(list(stuff.values()))  # ['ааа', 'ббб', 'ввв']